rag pipeline data integration to vector store


In [30]:
import os
from pathlib import Path
from IPython.display import display
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.config import Settings

In [31]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


# Function to read all PDF files from a directory
def read_all_pdfs(folder_path):
    all_documents = []

    pdf_dir = Path(folder_path)

    # Find only PDF files
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)

            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    print(f"\nTotal PDF pages loaded: {len(all_documents)}")

    return all_documents

In [32]:
all_pdf_documents = read_all_pdfs("../data")

Found 2 PDF files

Processing: cpi data for rag.pdf
Loaded 11 pages

Processing: GDP YEAR 24-25.pdf
Loaded 10 pages

Total PDF pages loaded: 21


In [33]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks = text_splitter.split_documents(documents)

    print(f"Pages received: {len(documents)}")
    print(f"Chunks created: {len(chunks)}")

    if chunks:
        print("\n--- First Chunk ---")
        print("Content:")
        print(chunks[0].page_content[:500])

        print("\nMetadata:")
        print(chunks[0].metadata)

    return chunks

In [34]:
chunks = split_documents(all_pdf_documents)

Pages received: 21
Chunks created: 60

--- First Chunk ---
Content:
This Press Release is embargoed against publication, telecast or circulation on internet till 4:00 PM of 12th January, 2026  
Page 1 of 11 
 
 
    
Price 
Statistics 
Division 
IS/ ISO 
9001: 2015 
CERTIFIED 
GOVERNMENT OF INDIA  
MINISTRY OF STATISTICS AND  PROGRAMME IMPLEMENTATION  
NATIONAL STATISTICS OFFICE 
 
Dated 12th January, 2026 
22 Pausa, Saka 1947 
PRESS RELEASE 
 
CONSUMER PRICE INDEX NUMBERS ON BASE 2012=100 FOR RURAL, 
URBAN AND COMBINED FOR THE MONTH OF DECEMBER, 2025 
I. Key hi

Metadata:
{'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-01-12T15:03:30+05:30', 'author': 'Mospi', 'moddate': '2026-01-12T15:49:42+05:30', 'source': '..\\data\\pdf\\cpi data for rag.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'cpi data for rag.pdf', 'file_type': 'pdf'}


In [35]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [36]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name

        print(f"Loading embedding model: {model_name}")

        self.model = SentenceTransformer(model_name)

        print(
            "Model loaded successfully."
        )

        print(
            f"Embedding dimension: "
            f"{self.model.get_embedding_dimension()}"
        )

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not texts:
            raise ValueError("No texts provided")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        print(f"Embedding shape: {embeddings.shape}")

        return embeddings

vector db

In [37]:
embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12942.28it/s]


Model loaded successfully.
Embedding dimension: 384


In [38]:
# Extract text from chunks
texts = [chunk.page_content for chunk in chunks]

# Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

Generating embeddings for 60 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.20s/it]

Embedding shape: (60, 384)


In [39]:
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)

Generating embeddings for 60 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.42s/it]

Embedding shape: (60, 384)


In [40]:
print("Number of chunks:", len(chunks))
print("Number of embeddings:", len(embeddings))
print("Embedding dimension:", embeddings.shape[1])

Number of chunks: 60
Number of embeddings: 60
Embedding dimension: 384


In [41]:
import os
import hashlib
from typing import List, Any

import chromadb
import numpy as np


class VectorStore:
    """Manages PDF document embeddings in ChromaDB."""

    def __init__(
        self,
        collection_name="pdf_documents",
        persist_directory="../data/vector_store"
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory

        # Create directory
        os.makedirs(
            self.persist_directory,
            exist_ok=True
        )

        # Persistent ChromaDB client
        self.client = chromadb.PersistentClient(
            path=self.persist_directory
        )

        # Create or load collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "PDF document embeddings for RAG",
                "hnsw:space": "cosine"
            }
        )

        print("Vector store initialized")
        print("Collection:", self.collection_name)
        print(
            "Database path:",
            os.path.abspath(self.persist_directory)
        )
        print(
            "Existing documents:",
            self.collection.count()
        )

    def _generate_document_id(self, document: Any) -> str:
        """
        Generate deterministic ID.

        Same PDF + page + content = same ID.
        """

        content = document.page_content

        source = document.metadata.get(
            "source_file",
            "unknown"
        )

        page = document.metadata.get(
            "page",
            "unknown"
        )

        unique_text = (
            f"{source}|{page}|{content}"
        )

        document_hash = hashlib.sha256(
            unique_text.encode("utf-8")
        ).hexdigest()

        return f"doc_{document_hash}"

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """Add or update documents in ChromaDB."""

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match "
                "number of embeddings"
            )

        if not documents:
            print("No documents to add.")
            return

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for doc, embedding in zip(
            documents,
            embeddings
        ):
            doc_id = self._generate_document_id(doc)

            metadata = dict(doc.metadata)

            metadata["content_length"] = len(
                doc.page_content
            )

            ids.append(doc_id)

            metadatas.append(metadata)

            documents_text.append(
                doc.page_content
            )

            embeddings_list.append(
                embedding.tolist()
            )

        # Upsert instead of add
        self.collection.upsert(
            ids=ids,
            embeddings=embeddings_list,
            metadatas=metadatas,
            documents=documents_text
        )

        print(
            f"Successfully added/updated "
            f"{len(documents)} documents."
        )

        print(
            "Total documents:",
            self.collection.count()
        )

    def clear_collection(self):
        """Delete all documents from the collection."""

        data = self.collection.get(
            include=[]
        )

        ids = data["ids"]

        if ids:
            self.collection.delete(ids=ids)
            print(f"Deleted {len(ids)} documents.")
        else:
            print("Collection is already empty.")

        print(
            "Remaining documents:",
            self.collection.count()
        )

    def get_collection_count(self):
        return self.collection.count()

In [42]:
vector_store = VectorStore()

Vector store initialized
Collection: pdf_documents
Database path: d:\rag\data\vector_store
Existing documents: 120


In [43]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vector_store.add_documents(chunks,embeddings)

Generating embeddings for 60 texts...


Batches: 100%|██████████| 2/2 [00:03<00:00,  1.58s/it]

Embedding shape: (60, 384)
Successfully added/updated 60 documents.
Total documents: 120


In [44]:
vector_store.clear_collection()

Deleted 120 documents.
Remaining documents: 0


In [45]:
vector_store = VectorStore()

Vector store initialized
Collection: pdf_documents
Database path: d:\rag\data\vector_store
Existing documents: 0


Retriever Pipeline From VectorStore

In [46]:
vector_store.add_documents(
    chunks,
    embeddings
)

Successfully added/updated 60 documents.
Total documents: 60


In [47]:
data = vector_store.collection.get(
    include=["documents", "metadatas"]
)

print("Total IDs:", len(data["ids"]))
print("Total documents:", len(data["documents"]))
print("Total metadata:", len(data["metadatas"]))

Total IDs: 60
Total documents: 60
Total metadata: 60


In [48]:
for i, doc_id in enumerate(data["ids"][:10]):
    print(i, doc_id)

0 doc_12b46f5e8ca87ad8d9c57f9769c8e2650b236fd21992ddbb9e27d0b1fbd893b5
1 doc_a38106dad1946cb2ecc5860224726edf6536af4c768ae4727f91bfc03665e163
2 doc_e05575a80cbed800ffb0b670c0139be7f41c98f2e4d398fd0c9926129ecafe27
3 doc_d6650380cfee441175ca9f24fd07178833cdc1a1d220062faf0f9d6441c8607f
4 doc_82f43cec5a5c19963c8b336d626ac7312930803ddbfb1d9f39f805bafe4a7654
5 doc_1b524ad743e6a96243225fdeff187f9d97a002cd5c178fdb7c9549b57ccd9443
6 doc_960876523ea6ea949af2237a27148b4484bce26fbd0bbbb31302d9bbaced2225
7 doc_41698cea466e2b71beba93e86aa3db15207b91e38b60a506e1744782ce711c8c
8 doc_9c119574f3836d28e9faa33dbabe889b61222571be0d964cd3bcf1f62aa0b106
9 doc_441b27319497c1b1a1a00f21367971002cd79ae3231ead3b1eff171ec274b3b9


In [49]:
class RAGRetriever:
    """Retrieves the most relevant PDF chunks from ChromaDB."""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:

        print(f"Query: {query}")
        print(f"Top K: {top_k}")

        # Generate embedding for the user's question
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        # Search ChromaDB
        results = self.vector_store.collection.query(
            query_embeddings=[
                query_embedding.tolist()
            ],
            n_results=top_k
        )

        retrieved_docs = []

        if not results["documents"] or not results["documents"][0]:
            print("No documents found.")
            return []

        documents = results["documents"][0]
        metadatas = results["metadatas"][0]
        distances = results["distances"][0]
        ids = results["ids"][0]

        for rank, (
            doc_id,
            document,
            metadata,
            distance
        ) in enumerate(
            zip(
                ids,
                documents,
                metadatas,
                distances
            ),
            start=1
        ):

            # Because our Chroma collection uses cosine distance
            similarity_score = 1 - distance

            if similarity_score >= score_threshold:

                retrieved_docs.append({
                    "id": doc_id,
                    "content": document,
                    "metadata": metadata,
                    "similarity_score": similarity_score,
                    "distance": distance,
                    "rank": rank
                })

        print(
            f"Retrieved {len(retrieved_docs)} documents."
        )

        return retrieved_docs

In [50]:
rag_retriever = RAGRetriever(
    vector_store,
    embedding_manager
)

In [51]:
results = rag_retriever.retrieve(
    "What is Annual GDP Estimates and Growth Rates?",
    top_k=5
)

Query: What is Annual GDP Estimates and Growth Rates?
Top K: 5
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 23.06it/s]

Embedding shape: (1, 384)
Retrieved 5 documents.


In [52]:

for result in results:
    print("\n" + "=" * 80)
    print("Rank:", result["rank"])
    print("ID:", result["id"])
    print("Source:", result["metadata"].get("source_file"))
    print("Page:", result["metadata"].get("page"))
    print("Similarity:", result["similarity_score"])
    print("Distance:", result["distance"])
    print("\nContent:")
    print(result["content"][:500])


Rank: 1
ID: doc_eb8e1d9733d9fc0bed08a1d0544c52f80b990b6d276d4ac3848e67bd11ebb8a0
Source: GDP YEAR 24-25.pdf
Page: 1
Similarity: 0.6367501616477966
Distance: 0.36324983835220337

Content:
components of GDP and Annual estimates of Gross/Net National Income and Per Capita Income for 
the Financial Years 2022-23, 2023-24 and 2024-25 at Constant and Current Prices are given 
in Statements 1 to 4 of Annexure A. 
Key Highlights: 
➢ Real GDP has been estimated to grow by 6.4% in FY 2024-25 as compared to the growth rate of
8.2% in Provisional Estimate (PE) of GDP for FY 2023-24. Nominal GDP has witnessed a growth
rate of 9.7% in FY 2024-25 over the growth rate of 9.6% in FY 2023-24.
➢ R

Rank: 2
ID: doc_d28105ed260de27b6f611d6ec1b80143086667ccc29a0282e9f7865eb617246b
Source: GDP YEAR 24-25.pdf
Page: 1
Similarity: 0.5742970705032349
Distance: 0.42570292949676514

Content:
This Press Release is embargoed against publication, telecast or circulation on internet till 4:00 pm on 7th January, 2025.

In [53]:
results = rag_retriever.retrieve(
    "What is the Consumer Price Index?",
    top_k=5
)

Query: What is the Consumer Price Index?
Top K: 5
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 75.16it/s]

Embedding shape: (1, 384)
Retrieved 5 documents.


In [54]:
for result in results:
    print("\n" + "=" * 80)
    print("Rank:", result["rank"])
    print("Source:", result["metadata"].get("source_file"))
    print("Page:", result["metadata"].get("page"))
    print("Similarity:", result["similarity_score"])
    print("Content:", result["content"][:400])


Rank: 1
Source: cpi data for rag.pdf
Page: 5
Similarity: 0.5220935344696045
Content: This Press Release is embargoed against publication, telecast or circulation on internet till 4:00 PM of 12th January, 2026  
Page 6 of 11 
 
 
    
Price 
Statistics 
Division 
IS/ ISO 
9001: 2015 
CERTIFIED 
         
              Annexure II 
All-India year-on-year inflation rates (%) for General, Group and Sub -group level CPI and CFPI 
numbers for December, 2025 (Provisional) for Rural, Urba

Rank: 2
Source: cpi data for rag.pdf
Page: 5
Similarity: 0.5196205377578735
Content: 6.1.03 Transport and 
communication 176.7 178.5 1.02 165.8 166.8 0.60 171.0 172.3 0.76 
 6.1.04 Recreation and 
amusement 181.5 183.3 0.99 176.7 179.0 1.30 178.8 180.9 1.17 
 6.1.05 Education 192.2 197.8 2.91 187.9 194.8 3.67 189.7 196.0 3.32 
 6.1.06 Personal care and 
effects 206.3 265.1 28.50 208.0 265.1 27.45 207.0 265.1 28.07 
6  Miscellaneous 190.8 203.4 6.60 182.0 192.2 5.60 186.5 198.0 6.1

Rank: 3
Source: cpi data 

In [55]:
rag_retriever

In [56]:
rag_retriever.retrieve("what is Annual GDP Estimates and Growth Rates?")


Query: what is Annual GDP Estimates and Growth Rates?
Top K: 5
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 56.93it/s]

Embedding shape: (1, 384)
Retrieved 5 documents.


[{'id': 'doc_eb8e1d9733d9fc0bed08a1d0544c52f80b990b6d276d4ac3848e67bd11ebb8a0',
  'content': 'components of GDP and Annual estimates of Gross/Net National Income and Per Capita Income for \nthe Financial Years 2022-23, 2023-24 and 2024-25 at Constant and Current Prices are given \nin Statements 1 to 4 of Annexure A. \nKey Highlights: \n➢ Real GDP has been estimated to grow by 6.4% in FY 2024-25 as compared to the growth rate of\n8.2% in Provisional Estimate (PE) of GDP for FY 2023-24. Nominal GDP has witnessed a growth\nrate of 9.7% in FY 2024-25 over the growth rate of 9.6% in FY 2023-24.\n➢ Real GVA has grown by 6.4% in FY 2024-25 over the growth rate of 7.2% in FY 2023-24.\nNominal GVA has shown a growth rate of 9.3% in FY 2024-25 as compared to the growth rate\nof 8.5% in FY 2023-24.\n➢ Real GVA of Agriculture and allied sector has been estimated to grow by 3.8% during 2024-25\nas compared to the growth of 1.4% witnessed during the last year that is 2023-24.\n➢ Real GVA of ‘Constru

Integration Vector Context pipeline With LLM output

In [67]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0.1,
)

response = llm.invoke(
    "Explain how an AI system answers questions using information retrieved from PDF documents."
)

print(response.content)

An AI system can answer questions using information retrieved from PDF documents through a process called Natural Language Processing (NLP) and Information Retrieval (IR). Here's a step-by-step explanation of how it works:

1. **Text Preprocessing**: The AI system first preprocesses the PDF document by extracting the text content. This involves removing unnecessary characters, such as headers, footers, and images, and converting the text to a digital format.
2. **Tokenization**: The preprocessed text is then tokenized, which involves breaking down the text into individual words or tokens. This allows the AI system to analyze each word separately.
3. **Part-of-Speech (POS) Tagging**: The AI system uses POS tagging to identify the grammatical category of each word, such as noun, verb, adjective, etc. This helps the system understand the context and meaning of the text.
4. **Named Entity Recognition (NER)**: The AI system uses NER to identify named entities, such as people, places, organi

In [68]:
def rag_simple(question, top_k=5):
    docs = rag_retriever.retrieve(question, top_k=top_k)

    if not docs:
        return {
            "question": question,
            "answer": "No relevant documents found.",
            "sources": []
        }

    context = "\n\n".join(doc["content"] for doc in docs)

    prompt = f"""Answer the question using only the context below.
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {question}

Answer:"""

    response = llm.invoke(prompt)

    sources = [
        f'{doc["metadata"].get("source_file")} (page {doc["metadata"].get("page_label")})'
        for doc in docs
    ]

    return {
        "question": question,
        "answer": response.content,
        "sources": sources
    }

In [69]:
result = rag_simple(
    "What is Annual GDP Estimates and Growth Rates?"
)

print("QUESTION:", result["question"])
print("\nANSWER:", result["answer"])
print("\nSOURCES:")

for source in result["sources"]:
    print(source)

Query: What is Annual GDP Estimates and Growth Rates?
Top K: 5
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 73.99it/s]

Embedding shape: (1, 384)
Retrieved 5 documents.


QUESTION: What is Annual GDP Estimates and Growth Rates?

ANSWER: Annual GDP Estimates for the Financial Year 2024-25 are given in the Press Note, along with its expenditure components both at Constant (2011-12) and Current Prices.

The growth rates mentioned are:

* Real GDP: 6.4% in FY 2024-25 (compared to 8.2% in FY 2023-24)
* Nominal GDP: 9.7% in FY 2024-25 (compared to 9.6% in FY 2023-24)
* Real GVA: 6.4% in FY 2024-25 (compared to 7.2% in FY 2023-24)
* Nominal GVA: 9.3% in FY 2024-25 (compared to 8.5% in FY 2023-24)

SOURCES:
GDP YEAR 24-25.pdf (page 2)
GDP YEAR 24-25.pdf (page 2)
GDP YEAR 24-25.pdf (page 6)
GDP YEAR 24-25.pdf (page 6)
GDP YEAR 24-25.pdf (page 7)


In [70]:
def rag_simple(question, top_k=4):
    # 1. Retrieve relevant chunks from your PDF documents
    results = rag_retriever.retrieve(question)

    # 2. Build context from the retrieved results
    context_parts = []

    for result in results[:top_k]:
        metadata = result["metadata"]
        source = metadata.get("source_file", "Unknown PDF")
        page = metadata.get("page", "Unknown")

        context_parts.append(
            f"Source: {source}, Page: {page}\n"
            f"{result['content']}"
        )

    context = "\n\n".join(context_parts)

    # 3. Ask Ollama to answer using the retrieved context
    prompt = f"""
You are a PDF research assistant.

Answer the question using only the provided context.
If the context does not contain the answer, say you could not find it
in the provided PDFs.
Mention the source and page when possible.

Context:
{context}

Question:
{question}

Answer:
"""

    response = llm.invoke(prompt)

    return {
        "question": question,
        "answer": response.content,
        "sources": [
            {
                "source": r["metadata"].get("source_file", "Unknown PDF"),
                "page": r["metadata"].get("page", "Unknown"),
                "similarity": r.get("similarity_score")
            }
            for r in results[:top_k]
        ]
    }

In [71]:
def rag_simple(question, top_k=4):
    # 1. Retrieve relevant chunks from your PDF documents
    results = rag_retriever.retrieve(question)

    # 2. Build context from the retrieved results
    context_parts = []

    for result in results[:top_k]:
        metadata = result["metadata"]
        source = metadata.get("source_file", "Unknown PDF")
        page = metadata.get("page", "Unknown")

        context_parts.append(
            f"Source: {source}, Page: {page}\n"
            f"{result['content']}"
        )

    context = "\n\n".join(context_parts)

    # 3. Ask Ollama to answer using the retrieved context
    prompt = f"""
You are a PDF research assistant.

Answer the question using only the provided context.
If the context does not contain the answer, say you could not find it
in the provided PDFs.
Mention the source and page when possible.

Context:
{context}

Question:
{question}

Answer:
"""

    response = llm.invoke(prompt)

    return {
        "question": question,
        "answer": response.content,
        "sources": [
            {
                "source": r["metadata"].get("source_file", "Unknown PDF"),
                "page": r["metadata"].get("page", "Unknown"),
                "similarity": r.get("similarity_score")
            }
            for r in results[:top_k]
        ]
    }

In [72]:
print(rag_retriever)

In [66]:
def chat_with_documents():
    print("Chat with your PDFs. Type 'exit' to stop.\n")

    while True:
        question = input("You: ").strip()

        if not question:
            continue

        if question.lower() in ("exit", "quit", "q"):
            print("Chat ended.")
            break

        try:
            result = rag_simple(question)
        except Exception as e:
            print(f"Error: {e}\n")
            continue

        print("\nAssistant:", result["answer"])

        print("\nSources:")
        for s in result["sources"]:
            print(f'  - {s["source"]} (page {s["page"]}) | similarity {s["similarity"]:.2f}')

        print("\n" + "-" * 60 + "\n")


chat_with_documents()

Chat with your PDFs. Type 'exit' to stop.

Query: hvhjh
Top K: 5
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.41it/s]

Embedding shape: (1, 384)
Retrieved 5 documents.



Assistant: I couldn't find the answer to your question in the provided PDFs. The text appears to be a table of CPI (Consumer Price Index) data for various states in India, but it does not contain any information about the value "hvhjh".

Sources:
  - cpi data for rag.pdf (page 2) | similarity 0.18
  - cpi data for rag.pdf (page 6) | similarity 0.18
  - cpi data for rag.pdf (page 6) | similarity 0.11
  - cpi data for rag.pdf (page 7) | similarity 0.10

------------------------------------------------------------

Query: Methodology and Major Data Sources
Top K: 5
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 66.54it/s]

Embedding shape: (1, 384)
Retrieved 5 documents.



Assistant: The methodology for the Advance Estimates of GDP is based on the benchmark-indicator method, which involves extrapolating the estimates available for the previous financial year (2023-24) using relevant indicators reflecting the performance of sectors.

The major data sources used for the compilation of these estimates include:

1. Index of Industrial Production (IIP)
2. Financial performance of Listed Companies based on available quarterly financial results for Q1 and Q2 of 2024-25
3. Estimates of Major Agricultural Crops and Horticultural crops for 2024-25 provided by the Ministry of Agriculture and Fisheries and Water Resources
4. Data from various Ministries/Departments/Private Agencies

Source: GDP YEAR 24-25.pdf, Page: 5

Sources:
  - GDP YEAR 24-25.pdf (page 5) | similarity 0.35
  - GDP YEAR 24-25.pdf (page 0) | similarity 0.33
  - cpi data for rag.pdf (page 9) | similarity 0.32
  - GDP YEAR 24-25.pdf (page 8) | similarity 0.32

--------------------------------------

Batches: 100%|██████████| 1/1 [00:00<00:00, 52.22it/s]

Embedding shape: (1, 384)
Retrieved 5 documents.



Assistant: According to the provided PDFs, the year-on-year growth rates (%) in main indicators of production of Rice are:

* November 2024: -1.11% (Source: cpi data for rag.pdf, Page: 9)
* December 2024: -1.26% (Source: cpi data for rag.pdf, Page: 9)
* November 2025: 2.51% (Source: cpi data for rag.pdf, Page: 9)
* December 2025: 2.51% (Source: cpi data for rag.pdf, Page: 9)

Note that the growth rates for November 2025 and December 2025 are provisional, as indicated on Page 9 of the cpi data for rag.pdf.

Sources:
  - GDP YEAR 24-25.pdf (page 5) | similarity 0.53
  - cpi data for rag.pdf (page 8) | similarity 0.46
  - GDP YEAR 24-25.pdf (page 1) | similarity 0.46
  - GDP YEAR 24-25.pdf (page 6) | similarity 0.44

------------------------------------------------------------

Chat ended.
